# Canadian Substance Use Dashboard
## Data Preparation Notebook

**Author:** Madina Alizada 

**Date:** May 2026  

**Data Source:** Health Canada Canadian Substance Use Survey (CSUS) 2023

### What this notebook does
This notebook takes three separate data files from Health Canada 
covering alcohol, cannabis, and opioids and combines them into 
one clean master table that Tableau can use to build our dashboard.

Each step is explained in plain English so anyone can follow along, 
no technical background needed!

## Step 1: Load the raw data files
We open the three CSV files downloaded from Health Canada. 
Each file covers one substance and contains survey results 
from 36,000+ Canadians aged 15 and older in 2023.

In [4]:
import pandas as pd
import os

alcohol  = pd.read_csv("../data/Alcohol.csv", encoding="utf-8-sig")
cannabis = pd.read_csv("../data/Cannabis.csv", encoding="utf-8-sig")
opioids  = pd.read_csv("../data/Opioids.csv", encoding="utf-8-sig")

print(f"Alcohol rows loaded:  {len(alcohol)}")
print(f"Cannabis rows loaded: {len(cannabis)}")
print(f"Opioids rows loaded:  {len(opioids)}")

Alcohol rows loaded:  1979
Cannabis rows loaded: 5600
Opioids rows loaded:  2738


## Step 2: Filter for age group rows only
Each file contains breakdowns by many categories like province, 
gender, income, and race. We only keep rows broken down by age 
group since that is the central question of our dashboard.

SQL equivalent: `WHERE disagg_var1 = 'Age'`

In [5]:
alcohol_age  = alcohol[alcohol['disagg_var1'] == 'Age'].copy()
cannabis_age = cannabis[cannabis['disagg_var1'] == 'Age'].copy()
opioids_age  = opioids[opioids['disagg_var1'] == 'Age'].copy()

print(f"Alcohol age rows:  {len(alcohol_age)}")
print(f"Cannabis age rows: {len(cannabis_age)}")
print(f"Opioids age rows:  {len(opioids_age)}")

Alcohol age rows:  127
Cannabis age rows: 366
Opioids age rows:  180


## Step 3: Add a substance label to each table
When we stack the three tables together we need a way to tell 
which row came from which file. We add a new column called 
substance to label every row as Alcohol, Cannabis, or Opioids.

SQL equivalent: `SELECT 'Alcohol' AS substance ...`

In [6]:
alcohol_age['substance']  = 'Alcohol'
cannabis_age['substance'] = 'Cannabis'
opioids_age['substance']  = 'Opioids'

print("Substance labels added successfully!")

Substance labels added successfully!


## Step 4: Keep only the columns Tableau needs
The original files have 22 columns each. Many are in French 
or contain technical notes we do not need for our charts. 
We keep only 6 columns that our dashboard actually uses.

SQL equivalent: `SELECT substance, question, disaggregate1, colpercent, numerator, denominator`

In [7]:
cols = ['substance', 'question', 'disaggregate1', 
        'colpercent', 'numerator', 'denominator']

alcohol_clean  = alcohol_age[cols].copy()
cannabis_clean = cannabis_age[cols].copy()
opioids_clean  = opioids_age[cols].copy()

print(f"Columns kept: {cols}")

Columns kept: ['substance', 'question', 'disaggregate1', 'colpercent', 'numerator', 'denominator']


## Step 5: Stack all three tables into one master table
We pile the three filtered tables on top of each other into 
one single table. This is exactly what SQL's UNION ALL command 
does. It combines rows from multiple tables that share the 
same column structure.

SQL equivalent: `... UNION ALL ... UNION ALL ...`

In [8]:
master = pd.concat(
    [alcohol_clean, cannabis_clean, opioids_clean],
    ignore_index=True
)

print(f"Master table total rows: {len(master)}")
master.head(10)

Master table total rows: 673


,substance,question,disaggregate1,colpercent,numerator,denominator
0,Alcohol,Lifetime use of alcohol,15-19,70.1,1820,2495.0
1,Alcohol,Lifetime use of alcohol,20-24,87.3,3917,4479.0
2,Alcohol,Lifetime use of alcohol,25-54,90.3,15335,16817.0
3,Alcohol,Lifetime use of alcohol,55+,93.2,11472,12244.0
4,Alcohol,Past 12 month use of alcohol,15-19,62.7,1635,2487.0
5,Alcohol,Past 12 month use of alcohol,20-24,81.0,3638,4477.0
6,Alcohol,Past 12 month use of alcohol,25-54,80.4,13505,16798.0
7,Alcohol,Past 12 month use of alcohol,55+,80.4,9747,12232.0
8,Alcohol,Past 30 day use of alcohol,15-19,42.6,1155,2483.0
9,Alcohol,Past 30 day use of alcohol,20-24,63.4,2865,4474.0


## STEP 6: Rename columns to  names that non-tech can understand
  SQL equivalent: SELECT question AS indicator,
                        disaggregate1 AS age_group ...

 What this means: the original column names from Health Canada
 are technical shorthand. We rename them so anyone opening
 the file immediately understands what each column means.

In [22]:

 
master.rename(columns={
    'question':      'indicator',
    'disaggregate1': 'age_group',
    'colpercent':    'percentage',
    'numerator':     'sample_count',
    'denominator':   'total_surveyed'
}, inplace=True)
 
print(f"  New column names: {list(master.columns)}")

  New column names: ['substance', 'indicator', 'age_group', 'percentage', 'sample_count', 'total_surveyed']


## STEP 7: Round percentage to 2 decimal places

 What this means: some percentage values have many decimal
 places like 42.342115. We round to 2 decimal places so
 the numbers look clean in Tableau tooltips and charts.

In [23]:
 
master['percentage'] = master['percentage'].round(2)

## STEP 8: Quality check before saving

 What this means: we print a summary of the final table to
 confirm everything looks correct before saving the file.
 This is like a final review before submitting work.

In [24]:

print(f"Total rows:         {len(master)}")
print(f"Total columns:      {len(master.columns)}")
print(f"Columns:            {list(master.columns)}")
print(f"Substances:         {list(master['substance'].unique())}")
print(f"Age groups:         {list(master['age_group'].unique())}")
print(f"\nNull values per column:")
print(master.isnull().sum().to_string())
print(f"\nSample of first 8 rows:")
print(master.head(8).to_string())

Total rows:         638
Total columns:      6
Columns:            ['substance', 'indicator', 'age_group', 'percentage', 'sample_count', 'total_surveyed']
Substances:         ['Alcohol', 'Cannabis', 'Opioids']
Age groups:         ['15-19', '20-24', '25-54', '55+']

Null values per column:
substance         0
indicator         0
age_group         0
percentage        0
sample_count      0
total_surveyed    0

Sample of first 8 rows:
  substance                     indicator age_group  percentage  sample_count  total_surveyed
0   Alcohol       Lifetime use of alcohol     15-19        70.1          1820          2495.0
1   Alcohol       Lifetime use of alcohol     20-24        87.3          3917          4479.0
2   Alcohol       Lifetime use of alcohol     25-54        90.3         15335         16817.0
3   Alcohol       Lifetime use of alcohol       55+        93.2         11472         12244.0
4   Alcohol  Past 12 month use of alcohol     15-19        62.7          1635          2487.0
5 

## STEP 9: Dropping the rows where the percentage is null 
These are rows Health Canada suppressed due to small sample sizes
 This is standard practice in public health data reporting
 ### Handling missing values
Some percentage values are missing because Health Canada 
intentionally did not report them. This happens when the 
number of survey respondents for a specific group was too 
small to calculate a reliable percentage. Reporting a 
percentage based on very few people would be misleading, 
so Health Canada left those cells blank. We remove these 
rows safely because none of them affect the charts we are 
building for our dashboard.



In [18]:
before = len(master)
master = master.dropna(subset=['percentage'])
after = len(master)

print(f"Rows before dropping nulls: {before}")
print(f"Rows after dropping nulls:  {after}")
print(f"Rows removed: {before - after}")
print(f"\nNull values remaining:")
print(master.isnull().sum())

Rows before dropping nulls: 638
Rows after dropping nulls:  638
Rows removed: 0

Null values remaining:
substance         0
indicator         0
age_group         0
percentage        0
sample_count      0
total_surveyed    0
dtype: int64


## STEP 10: Save the master table as a CSV file

 What this means: we save the final clean table as a new file
 called master_age_substance.csv inside the data folder.
 This is the file we connect directly to Tableau to build
 our dashboard. Think of it as the final cleaned dataset
 that all our charts will be built from.

In [25]:

 
os.makedirs("data", exist_ok=True)
output_path = "data/master_age_substance.csv"
master.to_csv(output_path, index=False)
 
print(f"\nClean master file saved to: {output_path}")
print(f"Total rows in final file: {len(master)}")


Clean master file saved to: data/master_age_substance.csv
Total rows in final file: 638
